# An RL agent: policy gradients → PPO

_Capstones_

**Stack four landmark reinforcement-learning (RL) papers into one working PPO agent that solves CartPole, then LunarLander.**

---

This notebook is a practice scaffold for the **An RL agent: policy gradients → PPO** lesson. We build the agent one labelled piece at a time — the REINFORCE policy, the A2C critic, GAE advantages, the clipped PPO update, and the rollout-collect-and-train loop — then run the clip-removal ablation that shows why PPO is stable.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders/dropdowns you can drag; without widgets they still run once with sensible defaults. Run each cell top to bottom, experiment with the knobs, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

## Setup

Colab has NumPy, pandas, matplotlib, and PyTorch preinstalled, but not Gymnasium. Keep the install cell below when running in Colab.

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab; gymnasium does not.
!pip install -q gymnasium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

CartPole is not a dataset; it is an environment. The "data" arrives as a stream of observations, actions, rewards, and done flags. Before training, inspect the observation dimensions, action choices, a random-policy baseline, and a tiny toy rollout that we will later turn into advantages.

### Step 1 — Imports, seeds, and the one task switch

PPO is on-policy, so we need a Gym environment plus a categorical policy. We start on `CartPole-v1` (solved at an average return of 475); switching `ENV_ID` to `LunarLander-v2` and bumping `UPDATES` runs the *same* code on the harder rocket-landing task.

`EPS = 0.2` is PPO's clip range — the width of the trust region that keeps each update from moving the policy too far.

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
# To switch to LunarLander later, ALSO run:  !pip install "gymnasium[box2d]"
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)
np.random.seed(0)

# The one task switch:
# Solve CartPole first; then set ENV_ID = "LunarLander-v2" and bump UPDATES to ~200.
# LunarLander also needs Box2D:  !pip install "gymnasium[box2d]"
ENV_ID  = "CartPole-v1"     # <- change to "LunarLander-v2" to land the rocket
SOLVED  = 475.0             # CartPole-v1 solved threshold (use ~200 for LunarLander)
UPDATES = 60                # bump to ~200 for LunarLander

EPS = 0.2                   # PPO clip range (Eq. 7)

### Step 2 — The environment table

For CartPole, the observation is four continuous numbers: cart position, cart velocity, pole angle, and pole angular velocity. The action space is discrete: push left or push right. That is the full interface our neural network sees.

In [ ]:
env = gym.make(ENV_ID)
obs, _ = env.reset(seed=0)

obs_names = ["cart position", "cart velocity", "pole angle", "pole angular velocity"]
if len(obs) != 4:
    obs_names = [f"observation {i}" for i in range(len(obs))]
obs_df = pd.DataFrame({
    "dimension": list(range(len(obs))),
    "meaning on CartPole": obs_names,
    "initial value (seed 0)": np.round(obs, 4),
})
action_df = pd.DataFrame({"action id": list(range(env.action_space.n)),
                          "meaning on CartPole": ["push left", "push right"][:env.action_space.n]})
print("environment:", ENV_ID)
display(obs_df)
display(action_df)
env.close()

### Step 3 — A random-policy baseline

Before learning, sample actions uniformly at random. This gives a baseline return to beat: CartPole usually falls quickly under random control, far below the solved threshold of 475.

In [ ]:
def random_policy_returns(env_id=ENV_ID, episodes=8, seed=0):
    env = gym.make(env_id)
    returns = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        done, total = False, 0.0
        while not done:
            action = env.action_space.sample()
            obs, rew, term, trunc, _ = env.step(action)
            done = term or trunc
            total += float(rew)
        returns.append(total)
    env.close()
    return returns

rand_returns = random_policy_returns()
rand_df = pd.DataFrame({"episode": range(len(rand_returns)), "random-policy return": rand_returns})
display(rand_df)
print("random-policy mean return:", round(float(np.mean(rand_returns)), 1), "vs solved threshold", SOLVED)

plt.figure(figsize=(6.5, 3))
plt.bar(rand_df["episode"], rand_df["random-policy return"], color="#ffb86b")
plt.axhline(SOLVED, color="#ff6b6b", ls="--", label="solved threshold")
plt.xlabel("episode"); plt.ylabel("return")
plt.title("random policy baseline")
plt.legend(); plt.show()

### Step 4 — A tiny rollout table

A PPO batch is just a rollout table: each row records one transition. For the hero operation below, we use a short hand-made rollout with rewards and value estimates so every arithmetic step is visible.

In [ ]:
toy_rewards = [1.0, 1.0, 1.0, 0.0]
toy_values  = [0.20, 0.35, 0.50, 0.10]
toy_dones   = [0.0, 0.0, 0.0, 1.0]
toy_last_v  = 0.0

toy_rollout_df = pd.DataFrame({
    "step t": range(len(toy_rewards)),
    "reward r_t": toy_rewards,
    "value V(s_t)": toy_values,
    "done?": toy_dones,
})
display(toy_rollout_df)

plt.figure(figsize=(6.5, 3))
plt.plot(toy_rollout_df["step t"], toy_rollout_df["reward r_t"], "o-", label="reward")
plt.plot(toy_rollout_df["step t"], toy_rollout_df["value V(s_t)"], "o-", label="value estimate")
plt.xlabel("step"); plt.title("toy rollout inputs")
plt.legend(); plt.show()

## Reference implementation — gymnasium + PyTorch (Colab)

Four papers become one PPO agent:

| Paper/component | Contributes | In our code |
|---|---|---|
| REINFORCE | stochastic policy gradient | actor head `pi` and sampled actions |
| A2C | value baseline and entropy bonus | critic head `v`, value loss, entropy term |
| GAE | low-variance advantages | `compute_gae` backward recursion |
| PPO | clipped trust-region surrogate | `min(r*A, clip(r)*A)` in `ppo_update` |

We will build the two heart pieces — GAE and PPO's clipped surrogate — on tiny concrete numbers before using them in the training loop.

### Step 5 — The actor-critic network

One shared MLP `body` feeds two heads: the **actor** `pi` outputs action logits (REINFORCE's policy), and the **critic** `v` outputs a single state value `V(s)` (A2C's value baseline).

`forward` returns a `Categorical` distribution over actions plus the value estimate, so a single call gives us everything the rollout and the loss need.

In [ ]:
# Policy + value networks (REINFORCE's policy + A2C's critic, Track B).
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=64):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh())
        self.pi = nn.Linear(hidden, n_act)   # action logits -> policy (actor)
        self.v  = nn.Linear(hidden, 1)       # state value V(s) -> critic

    def forward(self, x):
        h = self.body(x)
        dist = Categorical(logits=self.pi(h))
        value = self.v(h).squeeze(-1)
        return dist, value

### Step 6 — Inspect one untrained forward pass

The actor and critic read the same observation but answer different questions: "which action should I sample?" and "how much return do I expect from here?" The untrained numbers are random, but the shapes are already correct.

In [ ]:
env = gym.make(ENV_ID)
obs, _ = env.reset(seed=0)
torch.manual_seed(0)
demo_net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
with torch.no_grad():
    dist, value = demo_net(torch.as_tensor(obs, dtype=torch.float32))
    probs = dist.probs.numpy()

forward_df = pd.DataFrame({
    "item": ["observation", "action probabilities", "value estimate"],
    "shape/value": [str(tuple(obs.shape)), str(np.round(probs, 3).tolist()), round(float(value), 3)],
})
display(forward_df)

plt.figure(figsize=(5, 3))
plt.bar(range(env.action_space.n), probs, color="#79c0ff")
plt.xticks(range(env.action_space.n)); plt.ylim(0, 1)
plt.xlabel("action"); plt.ylabel("probability")
plt.title("untrained actor probabilities")
plt.show()
env.close()

### Step 7 — Parameter table + init sanity check

A small CartPole PPO agent has only a few thousand parameters. Most live in the shared body; the actor and critic heads are tiny. The random-policy return from Step 3 is the behavioural sanity check before training.

In [ ]:
groups = {}
for name, p in demo_net.named_parameters():
    comp = name.split(".")[0]
    groups[comp] = groups.get(comp, 0) + p.numel()
total_params = sum(groups.values())
param_df = pd.DataFrame({
    "component": list(groups),
    "parameters": list(groups.values()),
    "% of total": [round(100 * v / total_params, 1) for v in groups.values()],
})
display(param_df)
print("total trainable parameters:", f"{total_params:,}")
print("untrained/random-policy mean return from Step 3:", round(float(np.mean(rand_returns)), 1))

## The hero operation, part A — GAE built backward

Generalized Advantage Estimation turns raw rewards and value estimates into low-variance advantages. Walking backwards through the rollout, each step's TD error is

`delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)`

and the advantage recursively accumulates

`A_t = delta_t + gamma * lambda * A_{t+1}`.

The `done` mask zeros the bootstrap when an episode ended.

### Step 8 — Compute every GAE row by hand

Read this table from bottom to top: the final step has no future after `done`, then each earlier row adds discounted future advantage.

In [ ]:
def gae_table(rewards, values, dones, last_v, gamma=0.99, lam=0.95):
    values_ext = list(values) + [last_v]
    gae = 0.0
    rows = []
    for t in reversed(range(len(rewards))):
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values_ext[t + 1] * mask - values_ext[t]
        next_gae = gae
        gae = delta + gamma * lam * mask * gae
        rows.append({
            "t": t,
            "r_t": rewards[t],
            "V_t": values_ext[t],
            "V_next": values_ext[t + 1],
            "delta_t": delta,
            "A_next before row": next_gae,
            "A_t": gae,
        })
    return pd.DataFrame(rows[::-1])

gae_df = gae_table(toy_rewards, toy_values, toy_dones, toy_last_v)
display(gae_df.round(3))

plt.figure(figsize=(6.5, 3))
plt.bar(gae_df["t"], gae_df["A_t"], color="#7ee787")
plt.axhline(0, color="black", lw=0.7)
plt.xlabel("step t"); plt.ylabel("advantage")
plt.title("GAE advantages on toy rollout")
plt.show()

### Step 9 — Package GAE into the reusable function

The training loop will collect Python lists during rollout, then call this function to produce two tensors: normalized-ish advantages for the actor objective and returns for critic regression.

In [ ]:
# GAE: low-variance advantages (delta_t = r + gamma V' - V; A_t = sum (gamma lam)^l delta).
def compute_gae(rewards, values, dones, last_v, gamma=0.99, lam=0.95):
    adv = torch.zeros(len(rewards))
    gae = 0.0
    values = values + [last_v]
    for t in reversed(range(len(rewards))):
        mask = 1.0 - dones[t]                                    # 0 if episode ended at t
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae = delta + gamma * lam * mask * gae                   # GAE recursion
        adv[t] = gae
    returns = adv + torch.tensor(values[:-1])                    # value targets for the critic
    return adv, returns

adv_demo, ret_demo = compute_gae(toy_rewards, toy_values, toy_dones, toy_last_v)
display(pd.DataFrame({"t": range(len(adv_demo)),
                      "advantage": adv_demo.numpy().round(3),
                      "critic return target": ret_demo.numpy().round(3)}))

**🎛️ Play with it — gamma and lambda in GAE**

`gamma` controls how far rewards look into the future; `lambda` controls how much multi-step TD error is mixed in. **Try:** lower `lambda` toward 0, then raise it toward 1. **Watch:** the bars become more local or more future-looking. **Why it matters:** GAE trades bias for variance.

In [ ]:
def play(gamma=0.99, lam=0.95):
    df = gae_table(toy_rewards, toy_values, toy_dones, toy_last_v, gamma=gamma, lam=lam)
    display(df[["t", "delta_t", "A_t"]].round(3))
    plt.figure(figsize=(6.5, 3))
    plt.bar(df["t"], df["A_t"], color="#7ee787")
    plt.axhline(0, color="black", lw=0.7)
    plt.xlabel("step t"); plt.ylabel("advantage")
    plt.title("GAE bars")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             gamma=FloatSlider(value=0.99, min=0.80, max=1.00, step=0.01),
             lam=FloatSlider(value=0.95, min=0.00, max=1.00, step=0.05))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## The hero operation, part B — PPO's clipped surrogate

PPO compares the new policy to the old rollout policy with

`ratio r = pi_new(action|state) / pi_old(action|state)`.

The per-sample objective is `min(r*A, clip(r, 1-eps, 1+eps)*A)`. For positive advantages, PPO refuses to reward increasing the action probability beyond `1+eps`; for negative advantages, it refuses to reward decreasing it below `1-eps`. Those flat regions are PPO's trust region.

### Step 10 — Plot the clipped objective for positive and negative advantages

This is the explanatory centerpiece: the flat parts remove the incentive to move the probability ratio too far while reusing the same rollout for several epochs.

In [ ]:
def ppo_objective(ratio, advantage, eps=EPS):
    ratio = np.asarray(ratio)
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped), unclipped, clipped

ratios = np.linspace(0, 2, 401)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
for ax, adv_val, color in zip(axes, [1.0, -1.0], ["#7ee787", "#ff6b6b"]):
    obj, unc, clp = ppo_objective(ratios, adv_val, EPS)
    ax.plot(ratios, unc, color="#999999", ls="--", label="r * A")
    ax.plot(ratios, clp, color="#79c0ff", ls=":", label="clip(r) * A")
    ax.plot(ratios, obj, color=color, lw=2.5, label="min")
    ax.axvline(1 - EPS, color="black", lw=0.7)
    ax.axvline(1 + EPS, color="black", lw=0.7)
    ax.set_title(f"advantage {adv_val:+.0f}")
    ax.set_xlabel("ratio r")
    ax.set_ylabel("objective")
    ax.legend(fontsize=8)
plt.suptitle("PPO clipped surrogate")
plt.tight_layout(); plt.show()

**🎛️ Play with it — clip epsilon changes the flat region**

`epsilon` widens or narrows the trust region around ratio 1. **Try:** move `epsilon` from 0.05 to 0.4. **Watch:** the vertical clip boundaries spread apart, changing where the objective goes flat. **Why it matters:** too small can slow learning; too large can become unstable.

In [ ]:
def play(eps=0.20):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True)
    for ax, adv_val, color in zip(axes, [1.0, -1.0], ["#7ee787", "#ff6b6b"]):
        obj, unc, clp = ppo_objective(ratios, adv_val, eps)
        ax.plot(ratios, unc, color="#999999", ls="--", label="r * A")
        ax.plot(ratios, obj, color=color, lw=2.5, label="clipped objective")
        ax.axvline(1 - eps, color="black", lw=0.7)
        ax.axvline(1 + eps, color="black", lw=0.7)
        ax.set_title(f"A={adv_val:+.0f}, eps={eps:.2f}")
        ax.set_xlabel("ratio r"); ax.set_ylabel("objective")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, eps=FloatSlider(value=0.20, min=0.05, max=0.40, step=0.01))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — one ratio, one advantage**

A single sample can be clipped or not clipped depending on the ratio and the sign of its advantage. **Try:** set a large ratio with positive advantage, then a tiny ratio with negative advantage. **Watch:** the printed table shows which term PPO keeps. **Why it matters:** the sign determines which direction is dangerous.

In [ ]:
def play(r=1.30, sign="positive advantage"):
    A = 2.0 if sign == "positive advantage" else -2.0
    clipped_r = float(np.clip(r, 1 - EPS, 1 + EPS))
    unc = r * A
    clp = clipped_r * A
    obj = min(unc, clp)
    clipped = abs(obj - unc) > 1e-9
    display(pd.DataFrame({
        "quantity": ["ratio r", "advantage A", "clip(r)", "r*A", "clip(r)*A", "PPO min"],
        "value": [round(r, 3), A, round(clipped_r, 3), round(unc, 3), round(clp, 3), round(obj, 3)],
    }))
    print("clipped?", clipped, "--", "trust region active" if clipped else "inside useful region")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             r=FloatSlider(value=1.30, min=0.00, max=2.00, step=0.05),
             sign=Dropdown(options=["positive advantage", "negative advantage"], value="positive advantage"))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — clip on/off during batch reuse**

PPO takes several epochs over the same rollout. **Try:** switch the clip off. **Watch:** the raw objective keeps growing as ratios drift; with the clip on, the incentive flattens outside the trust region. **Why it matters:** multi-epoch reuse needs a brake.

In [ ]:
def play(use_clip=True):
    epoch = np.arange(8)
    drift_ratio = 1.0 + 0.10 * epoch
    A = 1.0
    if use_clip:
        obj = np.minimum(drift_ratio * A, np.clip(drift_ratio, 1 - EPS, 1 + EPS) * A)
        label = "clipped PPO"
    else:
        obj = drift_ratio * A
        label = "no clip"
    display(pd.DataFrame({"reuse epoch": epoch, "ratio": drift_ratio.round(2), "objective": obj.round(2)}))
    plt.figure(figsize=(6.5, 3))
    plt.plot(epoch, obj, "o-", color="#79c0ff")
    plt.xlabel("epoch on same rollout"); plt.ylabel("objective for A=+1")
    plt.title(label)
    plt.show()
    print("With clipping, the curve stops rewarding ratios above", 1 + EPS)

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, use_clip=Checkbox(value=True))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 11 — Package the PPO update

We normalise advantages, recompute the new log-probabilities, form the ratio, and optimize the clipped surrogate plus A2C's value loss and entropy bonus. The `use_clip=False` branch is the ablation: raw surrogate, no trust region.

In [ ]:
# PPO update: A2C actor-critic loss (value + entropy) wrapped in PPO's clip (Eq. 7/9).
def ppo_update(net, opt, obs, acts, old_logp, adv, returns,
               clip_eps=EPS, c1=0.5, c2=0.01, epochs=10, use_clip=True):
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)                # normalize advantages
    for _ in range(epochs):                                      # reuse the batch -- safe via the clip
        dist, value = net(obs)
        new_logp = dist.log_prob(acts)
        ratio = torch.exp(new_logp - old_logp)                   # r_t = pi_theta / pi_theta_old
        if use_clip:
            unc = ratio * adv
            clp = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
            policy_loss = -torch.min(unc, clp).mean()            # PPO clipped surrogate (Eq. 7)
        else:
            policy_loss = -(ratio * adv).mean()                  # ABLATION: raw surrogate, no clip
        value_loss = (returns - value).pow(2).mean()             # A2C value loss
        entropy = dist.entropy().mean()                          # A2C entropy bonus
        loss = policy_loss + c1 * value_loss - c2 * entropy      # combined objective (Eq. 9)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)          # gradient clip (separate from L^CLIP)
        opt.step()

**🎛️ Play with it — entropy coefficient and exploration**

The entropy bonus rewards a spread-out action distribution. **Try:** increase `c2`. **Watch:** the total loss gives more credit to high entropy. **Why it matters:** early exploration helps the agent find balancing behaviours before it commits.

In [ ]:
def play(c2=0.01):
    p = np.linspace(0.01, 0.99, 99)
    entropy = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    bonus = c2 * entropy
    plt.figure(figsize=(6.5, 3))
    plt.plot(p, bonus, color="#c89bff")
    plt.xlabel("P(action 1)"); plt.ylabel("entropy bonus c2*H")
    plt.title("entropy encourages exploration")
    plt.show()
    print(f"max bonus at p=0.5: {c2 * np.log(2):.4f}; near-deterministic bonus: {bonus[0]:.4f}")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, c2=FloatSlider(value=0.01, min=0.00, max=0.10, step=0.005))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — discount gamma as a planning horizon**

Discounting weights a reward `k` steps away by `gamma^k`. **Try:** lower `gamma`. **Watch:** future rewards fade faster. **Why it matters:** CartPole needs enough horizon to value actions that keep the pole balanced later, not just immediately.

In [ ]:
def play(gamma=0.99):
    k = np.arange(40)
    weights = gamma ** k
    plt.figure(figsize=(6.5, 3))
    plt.bar(k, weights, color="#7ee787")
    plt.xlabel("steps in future k"); plt.ylabel("weight gamma^k")
    plt.title("discount horizon")
    plt.show()
    half = np.log(0.5) / np.log(gamma) if gamma < 1 else np.inf
    print("half-life in steps:", "infinite" if np.isinf(half) else round(float(half), 1))

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, gamma=FloatSlider(value=0.99, min=0.80, max=1.00, step=0.01))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 12 — One rollout plus one PPO update

Before the full run, collect a tiny on-policy rollout and take one PPO update. This is the whole algorithm compressed into one cell: collect observations/actions/log-probs, compute GAE, then update the network.

In [ ]:
def collect_rollout(env, net, obs, steps_per):
    O, Ac, LP, R, V, D = [], [], [], [], [], []
    ep_ret = 0.0
    finished_returns = []
    for _ in range(steps_per):
        ot = torch.as_tensor(obs, dtype=torch.float32)
        with torch.no_grad():
            dist, v = net(ot)
            a = dist.sample()
            lp = dist.log_prob(a)                                # log pi_old recorded at collection
        nobs, rew, term, trunc, _ = env.step(int(a))
        done = term or trunc
        O.append(ot); Ac.append(a); LP.append(lp)
        R.append(float(rew)); V.append(float(v)); D.append(1.0 if done else 0.0)
        ep_ret += float(rew)
        obs = nobs
        if done:
            finished_returns.append(ep_ret)
            ep_ret = 0.0
            obs, _ = env.reset()
    return obs, (O, Ac, LP, R, V, D), finished_returns

torch.manual_seed(0)
env = gym.make(ENV_ID)
one_net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
one_opt = torch.optim.Adam(one_net.parameters(), lr=3e-4)
obs, _ = env.reset(seed=0)
obs, batch, finished = collect_rollout(env, one_net, obs, steps_per=64)
O, Ac, LP, R, V, D = batch
with torch.no_grad():
    last_v = float(one_net(torch.as_tensor(obs, dtype=torch.float32))[1])
adv, ret = compute_gae(R, V, D, last_v)
before_value_mean = float(torch.tensor(V).mean())
ppo_update(one_net, one_opt, torch.stack(O), torch.stack(Ac), torch.stack(LP), adv, ret, epochs=2, use_clip=True)
print("collected steps:", len(R), "finished episodes:", len(finished))
print("mean reward in tiny rollout:", round(float(np.mean(R)), 3))
print("mean value estimate before update:", round(before_value_mean, 3))
env.close()

### Step 13 — The rollout + train loop

Each update first **collects a rollout**: acting on-policy, we record observations, actions, the log-probabilities under the *old* policy, rewards, values, and done flags. After `steps_per` steps we bootstrap the final value, compute GAE, and call `ppo_update` to improve the policy on that batch.

We track the average return over the last 20 episodes and stop once it crosses `SOLVED`. The `use_clip` flag is threaded through so the same loop can run the ablation.

In [ ]:
# Train until solved; PRINT the return rising.
def train(use_clip=True, updates=UPDATES, steps_per=2048):
    torch.manual_seed(0)
    env = gym.make(ENV_ID)
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(net.parameters(), lr=3e-4)
    obs, _ = env.reset(seed=0)
    ep_ret = 0.0
    recent, history = [], []
    max_updates = int(updates)
    for update in range(200):
        if update >= max_updates:
            break
        O, Ac, LP, R, V, D = [], [], [], [], [], []
        for _ in range(steps_per):                               # collect a rollout (ON-POLICY)
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                dist, v = net(ot)
                a = dist.sample()
                lp = dist.log_prob(a)                            # log pi_old recorded at collection
            nobs, rew, term, trunc, _ = env.step(int(a))
            done = term or trunc
            O.append(ot)
            Ac.append(a)
            LP.append(lp)
            R.append(float(rew))
            V.append(float(v))
            D.append(1.0 if done else 0.0)
            ep_ret += rew
            obs = nobs
            if done:
                recent.append(ep_ret)
                ep_ret = 0.0
                obs, _ = env.reset()
        with torch.no_grad():
            last_v = float(net(torch.as_tensor(obs, dtype=torch.float32))[1])
        adv, ret = compute_gae(R, V, D, last_v)
        ppo_update(net, opt, torch.stack(O), torch.stack(Ac), torch.stack(LP),
                   adv, ret, epochs=10, use_clip=use_clip)
        avg = sum(recent[-20:]) / max(1, len(recent[-20:]))      # avg return, last 20 episodes
        history.append(avg)
        print(f"  update {update:2d}  avg return (last 20 eps): {avg:6.1f}")
        recent = recent[-20:]
        if avg >= SOLVED:
            print(f"  -> SOLVED {ENV_ID}.")
            break
    env.close()
    return history

### Step 14 — Run full PPO, then the clip-removal ablation

We train with the clip on (full PPO), then again with `use_clip=False` — same network, GAE, learning rate, epochs and seed, only the trust region removed. Comparing the two return trajectories isolates the clipped surrogate as PPO's load-bearing contribution.

Clipped PPO should climb toward ~500 and stay solved; the no-clip ablation tends to rise then destabilise. Switch `ENV_ID` to `LunarLander-v2` to run the same code on the rocket. (Our small run, not a paper's reported number; values vary by hardware and seed.)

In [ ]:
print(f"Full PPO on {ENV_ID} (clipped surrogate, GAE, actor-critic + entropy):")
clip_hist = train(use_clip=True)
print("\nABLATION -- clip removed (raw r_t*A_t, same net / GAE / lr / epochs / seed):")
noclip_hist = train(use_clip=False)
print("\nClipped avg-return trajectory:", [round(h, 1) for h in clip_hist])
print("No-clip avg-return trajectory:", [round(h, 1) for h in noclip_hist])
# Clipped PPO climbs toward ~500 and stays there (SOLVED); the no-clip ablation rises
# then destabilizes. Exact numbers vary by hardware/seed -- our small run, not a paper's.
# Switch ENV_ID to "LunarLander-v2" (and !pip install "gymnasium[box2d]", updates ~200)
# to run the SAME code on the rocket-landing task.

## Visualize the data & results

_As the assembled PPO agent trains with GAE, actor-critic losses, entropy, and the clipped surrogate, does the average return rise, and what changes when we remove the clip?_ We answer with the histories recorded above — no retraining needed.

### Step 15 — Return curves: full PPO vs no clip

Both runs use the same seed and code path; only `use_clip` changes. The curve is noisy because RL data is generated by the current policy, not sampled from a fixed dataset.

In [ ]:
plt.figure(figsize=(7.5, 3.8))
plt.plot(range(len(clip_hist)), clip_hist, "o-", label="full PPO (clipped)", color="#7ee787")
plt.plot(range(len(noclip_hist)), noclip_hist, "o-", label="no-clip ablation", color="#ff6b6b")
plt.axhline(SOLVED, color="#999999", ls="--", label="solved threshold")
plt.xlabel("update"); plt.ylabel("avg return, last 20 eps")
plt.title("PPO vs no clip")
plt.legend(); plt.show()

result_df = pd.DataFrame({
    "run": ["full PPO (clipped)", "no-clip ablation"],
    "updates recorded": [len(clip_hist), len(noclip_hist)],
    "final avg return (our run)": [round(clip_hist[-1], 1), round(noclip_hist[-1], 1)],
    "best avg return (our run)": [round(max(clip_hist), 1), round(max(noclip_hist), 1)],
})
display(result_df)

### Step 16 — The clip plot explains the ablation

The ablation removes exactly the flat regions in the clipped-surrogate plot. Without those flats, repeated epochs on the same rollout can keep rewarding ever-larger probability ratios.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
obj, unc, clp = ppo_objective(ratios, advantage=1.0, eps=EPS)
ax.plot(ratios, unc, color="#ff6b6b", ls="--", label="no clip: r*A")
ax.plot(ratios, obj, color="#7ee787", lw=2.5, label="PPO: min clipped")
ax.axvspan(1 - EPS, 1 + EPS, color="#79c0ff", alpha=0.15, label="trust region")
ax.set_xlabel("ratio r"); ax.set_ylabel("objective, A=+1")
ax.set_title("clip removes far-ratio incentive")
ax.legend(); plt.show()

### Step 17 — Data recap: observations, actions, advantages

The final table ties the whole walkthrough together: the environment interface, the random baseline, the toy advantages, and the training histories are all different views of the same PPO pipeline.

In [ ]:
recap_df = pd.DataFrame({
    "thing inspected": ["observation dimensions", "actions", "random baseline mean", "toy GAE first advantage", "full PPO final return"],
    "value": [len(obs_df), env.action_space.n if 'env' in globals() else 2,
              round(float(np.mean(rand_returns)), 1), round(float(adv_demo[0]), 3),
              round(float(clip_hist[-1]), 1)],
})
display(recap_df)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In the clipped objective, suppose the probability ratio is $r_t = 1.6$ and the advantage is
            $\hat{A}_t = +2$, with clip range $\epsilon = 0.2$. What value does PPO's per-sample objective
            $\min(r_t \hat{A}_t,\ \text{clip}(r_t, 1-\epsilon, 1+\epsilon)\,\hat{A}_t)$ take, and why does
            this cap the update?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Compute the unclipped term: $r_t \hat{A}_t = 1.6 \times 2 = 3.2$. — _This is the raw surrogate gain &mdash; what REINFORCE/A2C would chase, unbounded._
- Clip the ratio: $\text{clip}(1.6,\ 0.8,\ 1.2) = 1.2$, then $1.2 \times 2 = 2.4$. — _Because the advantage is positive, the ratio is pinned at the upper edge $1+\epsilon = 1.2$._
- Take the min: $\min(3.2,\ 2.4) = 2.4$. — _The min discards the larger unclipped term, so the gradient through this sample is zeroed once the
                   ratio leaves the trust region &mdash; no incentive to push the policy further on this batch._

**Answer:** $2.4$. The clip removes the gradient once $r_t$ exceeds $1+\epsilon$, which is exactly what lets
                 PPO reuse the same rollout for several epochs without the policy blowing up.

</details>

**Problem 2.** The final CODE cell runs an ablation: it trains the identical agent with the clip removed
            (raw $r_t \hat{A}_t$, same net / GAE / learning rate / epochs / seed). Predict what the return curve
            does, and name which paper's contribution the ablation deletes.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Identify the deleted piece: removing the clip deletes PPO's (2017) contribution, leaving an A2C-style
                 surrogate that is reused for several epochs per batch. — _A2C is on-policy and meant for roughly one update per rollout; reusing a batch many times without a
                   trust region lets a single high-ratio action dominate._
- Predict the curve: it rises at first, then oscillates and crashes repeatedly instead of
                 settling near 500. — _Over-large updates on reused data push some action's ratio huge, breaking the policy; it must
                   re-learn, so the average return spikes and collapses._

**Answer:** The no-clip curve rises then oscillates/crashes; it deletes PPO's clipped surrogate objective. The
                 clip is what makes multi-epoch batch reuse safe.

</details>